In [1]:
import warnings
warnings.filterwarnings("ignore")

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

from pyspark.sql.types import *
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local-cluster[3,2,2048]")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print(sc.getConf().get("spark.driver.memory", "NOT SET"))

2g


In [2]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)
spark.conf.set("spark.sql.join.preferSortMergeJoin", "false")

In [3]:
my_schema_1 = """
VendorID INT,
tpep_pickup_datetime TIMESTAMP,
tpep_dropoff_datetime TIMESTAMP,
passenger_count DOUBLE,
trip_distance DOUBLE,
RatecodeID DOUBLE,
store_and_fwd_flag STRING,
PULocationID INT,
DOLocationID INT,
payment_type INT,
fare_amount DOUBLE,
extra DOUBLE,
mta_tax DOUBLE,
tip_amount DOUBLE,
tolls_amount DOUBLE,
improvement_surcharge DOUBLE,
total_amount DOUBLE,
congestion_surcharge DOUBLE,
Airport_fee DOUBLE,
cbd_congestion_fee DOUBLE
"""

my_schema_2 = """
LocationID INT,
Borough STRING, 
Zone STRING,
service_zone STRING
"""

In [4]:
df1 = spark.read.format('csv').option('header', 'true').schema(my_schema_1).load('yellow_tripdata_combined.csv')

In [5]:
df2 = spark.read.format('csv').option('header', 'true').schema(my_schema_2).load('taxi_lookup.csv')

In [6]:
df_shuffle = df1.hint("shuffle_hash").join(df2.hint("shuffle_hash"), df1['PULocationID'] == df2['LocationID'], how="inner")

spark.sparkContext.setJobDescription('Shuffle hash join')
df_shuffle.write.format('noop').mode('overwrite').save()

In [7]:
spark.conf.set('spark.sql.join.preferSortMergeJoin', 'true')

In [8]:
df_smj = df1.join(df2, df1['PULocationID']==df2['LocationID'], how='inner')

spark.sparkContext.setJobDescription('Sort Merge join')
df_smj.write.format('noop').mode('overwrite').save()